# 04 — Statistical Baseline (Phase 2)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name=='notebooks' else str(Path.cwd()))
from src.utils.data_loader import DatasetSource
from src.ml.baseline import load_series, add_event_labels, zscore_detect, evaluate
import pandas as pd

src = DatasetSource()
ei = src.read_csv('Wind Farm A/event_info.csv', sep=';')
print('events:', len(ei))


In [ ]:
# Pilot feature set (Farm A)
FEATURES = ['sensor_0_avg','wind_speed_3_avg','sensor_5_avg','power_30_avg','sensor_11_avg','sensor_13_avg']

# Anomaly example: event 0 (asset 0)
df = load_series(src, 'A', 0, asset_id=0)
df = add_event_labels(df, ei, 0)
print('shape', df.shape, 'anomaly rows', int(df['is_anomaly'].sum()))
print('label', df['event_label'].iloc[0])


In [ ]:
# Z-score baseline (threshold 3)
df = zscore_detect(df, FEATURES, threshold=3.0)
pred = df[df['train_test']=='prediction']
metrics = evaluate(pred['is_anomaly'].values, pred['is_pred_anomaly'].values, pred['time_stamp'])
print(metrics)


In [ ]:
# Normal example: event 24 (asset 0)
dfn = load_series(src, 'A', 24, asset_id=0)
dfn = add_event_labels(dfn, ei, 24)
dfn = zscore_detect(dfn, FEATURES, threshold=3.0)
predn = dfn[dfn['train_test']=='prediction']
metricsn = evaluate(predn['is_anomaly'].values, predn['is_pred_anomaly'].values, predn['time_stamp'])
print('normal series metrics:', metricsn)


In [ ]:
# Threshold sweep
import numpy as np
rows=[]
for thr in [2.0,2.5,3.0,3.5,4.0]:
    d = zscore_detect(df.copy(), FEATURES, threshold=thr)
    p = d[d['train_test']=='prediction']
    m = evaluate(p['is_anomaly'].values, p['is_pred_anomaly'].values, p['time_stamp'])
    rows.append({'threshold':thr, **m})
print(pd.DataFrame(rows).to_string(index=False))
